# Hướng Dẫn Giải Thích Chi Tiết: `src/ai_models.py`

Notebook này phân tích cấu trúc, công dụng và cách hoạt động của mô hình tối ưu **XGBoost** và mô hình mạng nơ-ron **Transformer** được xây dựng trong `src/ai_models.py`.

---

## 🔍 1. Tổng Quan Về Các Mô Hình Sử Dụng

Trong dự án này, chúng ta kết hợp hai trường phái mạnh mẽ nhất hiện nay trong Machine Learning:
1. **XGBoost (Cây Quyết Định Tối Ưu):** Cực kỳ mạnh mẽ đối với dữ liệu dạng bảng, tìm kiếm các mối quan hệ phi tuyến tính dựa trên các chỉ báo kỹ thuật một cách trực tiếp.
2. **Transformer (Deep Learning):** Kiến trúc tiên tiến nhất được kế thừa từ xử lý ngôn ngữ tự nhiên (NLP) áp dụng cho dữ liệu chuỗi thời gian, giúp mô hình tập trung sự chú ý (Attention) vào các phiên giao dịch có tác động lớn nhất trong lịch sử 45 ngày.

In [ ]:
import sys
import os
# Thêm thư mục gốc vào đường dẫn hệ thống để import src
sys.path.append(os.path.abspath('..'))

from src.ai_models import build_xgboost_optimized, build_transformer
print("Import các hàm khởi tạo mô hình thành công!")

## ⚙️ 2. Mô Hình XGBoost Tối Ưu Hóa Tìm Kiếm Siêu Tham Số

### Hàm `build_xgboost_optimized(X_train, y_train)`

XGBoost không nhận đầu vào dạng 3D chuỗi thời gian. Vì vậy, trước khi đưa vào huấn luyện, chúng ta làm phẳng dữ liệu (`reshape` từ 3D sang 2D):
$$\text{Shape: } [N, 45, 14] \rightarrow [N, 45 \times 14] = [N, 630]$$
Tức là 630 cột đặc trưng biểu thị tất cả giá trị chỉ báo của 45 ngày liên tiếp.

Hàm thực hiện tìm kiếm lưới (`GridSearchCV`) kết hợp phân chia chuỗi thời gian (`TimeSeriesSplit`) để tìm bộ tham số tốt nhất:
- `n_estimators`: Số lượng cây quyết định (100 hoặc 200).
- `max_depth`: Độ sâu tối đa của cây (3 hoặc 5).
- `learning_rate`: Tốc độ học (0.05 hoặc 0.1).
- `subsample` và `colsample_bytree`: Tỷ lệ mẫu và tỷ lệ đặc trưng dùng để xây dựng mỗi cây (chống quá khớp - Overfitting).
- `n_jobs=-1`: Chạy song song tìm kiếm trên tất cả các nhân CPU.

In [ ]:
# Tạo dữ liệu ngẫu nhiên giả lập tập huấn luyện để chạy thử nghiệm XGBoost
np_random = np.random.RandomState(42)
X_train_dummy = np_random.normal(size=(100, 45, 14))
y_train_dummy = np_random.normal(size=(100,))

# Làm phẳng đầu vào cho XGBoost
X_train_dummy_flat = X_train_dummy.reshape(X_train_dummy.shape[0], -1)
print(f"Hình dạng tập train dummy làm phẳng: {X_train_dummy_flat.shape}")

# Thử nghiệm chạy GridSearchCV trên dữ liệu dummy
xgb_dummy_model = build_xgboost_optimized(X_train_dummy_flat, y_train_dummy)

## 🤖 3. Kiến Trúc Mạng Nơ-ron Transformer Nâng Cấp

### Hàm `build_transformer(input_shape)`

Kiến trúc Transformer của chúng ta nhận đầu vào 3D trực tiếp mà không cần làm phẳng dữ liệu, cấu thành từ các thành phần sau:

1. **Lớp Đầu Vào (Input Layer):** Nhận ma trận `(time_steps=45, features=14)`.
2. **Nhúng Vị Trí Thời Gian (Temporal/Sinusoidal Position Encoding):** Thêm thông tin về thứ tự tuần tự của ngày giao dịch vào dữ liệu nhúng (ngày gần nhất, ngày xa nhất trong 45 phiên).
3. **Hai lớp Multi-Head Attention (8 heads, key_dim=128):** 
   - Giúp mạng nơ-ron tự học cách tập trung trọng số vào các thời điểm biến động mạnh trong lịch sử 45 ngày để đưa ra dự đoán giá tốt nhất.
4. **Residual Connections (Kết nối tắt) & Layer Normalization:** 
   - Giúp truyền trực tiếp thông tin từ các lớp trước ra lớp sau mà không lo suy giảm gradient (vanishing gradient), giúp huấn luyện mạng sâu ổn định hơn.
5. **Global Average Pooling & Dense Layers (Fully Connected):** 
   - Gom các chiều thời gian lại và kết nối với các lớp ẩn Dense (Dropout 20%) để đưa ra một giá trị thực số duy nhất (tỷ lệ thay đổi giá mở cửa).

In [ ]:
# Khởi tạo cấu trúc Transformer
transformer_shape = (45, 14)
transformer_model = build_transformer(transformer_shape)

# Hiển thị sơ đồ kiến trúc chi tiết
transformer_model.summary()